# RQ1: Attribution Stability Benchmark Under Meaning-Preserving Perturbations
Full-scale evaluation of token attribution robustness across 16 lexical, syntactic, and discourse perturbations on SST-2 (supports both **local RTX 4090** with optional Google Drive for Desktop sync and **Google Colab A100**).

Pipeline outputs:
- **Stability metrics:** Spearman $\rho$, Kendall $\tau$, Pearson $r$, Top-$k$ Jaccard ($k \in \{1, 3, 5\}$), plus Monte Carlo Noise Floor (`reseed_same_text`)
- **Decision Preservation & Flip Rate:** `rq1_perturbation_decisions.csv` and `rq1_preservation_summary.csv`
- **Perturbation Strength Analysis:** `rq1_strength_analysis.csv` (`abs_delta_p`, `char_edit_ratio`, `token_len_rel_diff`)
- **RQ2 Attribution Cache:** `rq1_attributions.jsonl` (original attributions with model & explainer metadata so RQ2 skips recomputing XAI)

> **IG Baseline Note:** Use `ig_baseline = "mean"` for new model runs (`Qwen/Qwen2.5-3B`, `Qwen/Qwen3-4B-Base`, `Qwen/Qwen3-8B-Base`). Only use `ig_baseline = "zero"` if resuming the legacy `Qwen/Qwen2.5-7B` run.

### 1. GPU Environment Verification

In [ ]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB) | bf16: {torch.cuda.is_bf16_supported()}")

### 2. Google Drive Mounting, Repository Setup, and Dependencies

In [ ]:
import os

try:
    IN_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IN_COLAB = False
IN_COLAB_HOSTED = IN_COLAB and os.path.isdir("/content")

# Optional: custom directory (e.g. Google Drive for Desktop on Windows: r"G:\My Drive\llm_xai_results")
LOCAL_DRIVE_RESULTS = "" #@param {type:"string"}

if IN_COLAB_HOSTED:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS = "/content/drive/MyDrive/llm_xai_results"
    os.makedirs(DRIVE_RESULTS, exist_ok=True)

    GITHUB_TOKEN = "" #@param {type:"string"}
    REPO_OWNER_REPO = "jakubstachyra/llm-xai" #@param {type:"string"}
    BRANCH = "main" #@param {type:"string"}

    try:
        from google.colab import userdata
        token_from_secrets = userdata.get('GH_TOKEN')
        if token_from_secrets:
            GITHUB_TOKEN = token_from_secrets
    except Exception:
        pass

    if GITHUB_TOKEN.strip():
        CLONE_URL = f"https://{GITHUB_TOKEN.strip()}@github.com/{REPO_OWNER_REPO}.git"
    else:
        CLONE_URL = f"https://github.com/{REPO_OWNER_REPO}.git"

    REPO_DIR = "/content/llm-xai"
    if not os.path.exists(REPO_DIR):
        res = !git clone -b {BRANCH} {CLONE_URL} {REPO_DIR}
        print("\n".join(res))
        if not os.path.exists(REPO_DIR):
            raise RuntimeError("Cloning error! Provide Personal Access Token (PAT) in GITHUB_TOKEN or set GH_TOKEN secret.")
        %cd {REPO_DIR}
    else:
        %cd {REPO_DIR}
        !git checkout {BRANCH}
        !git pull origin {BRANCH}

    !pip install -q "transformers>=4.51.0,<5.0.0" "accelerate>=0.30.0" "datasets>=2.19.0" "shap>=0.45.1" scikit-learn scipy pandas matplotlib seaborn huggingface_hub

    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            os.environ["HF_TOKEN"] = hf_token
    except Exception:
        pass
else:
    if LOCAL_DRIVE_RESULTS.strip():
        DRIVE_RESULTS = LOCAL_DRIVE_RESULTS.strip()
    else:
        drive_candidates = [
            r"G:\My Drive",
            r"G:\Mój dysk",
            r"D:\My Drive",
            r"D:\Mój dysk",
            os.path.expanduser("~/Google Drive/My Drive"),
            os.path.expanduser("~/Google Drive/Mój dysk"),
        ]
        detected_drive = next((p for p in drive_candidates if os.path.isdir(p)), None)
        DRIVE_RESULTS = os.path.join(detected_drive, "llm_xai_results") if detected_drive else "results"
    os.makedirs(DRIVE_RESULTS, exist_ok=True)
    mode_lbl = "Colab Local Runtime (RTX 4090 / local GPU)" if IN_COLAB else "Local Python / Jupyter"
    print(f"Running in: {mode_lbl}. Checkpoints & results directory: {os.path.abspath(DRIVE_RESULTS)}")

### 3. Pipeline Smoke Test (2 samples)
Quick verification that the selected Qwen model, tokenizer, perturbations, and all 5 explainers execute cleanly.

In [ ]:
smoke_model = "Qwen/Qwen2.5-3B" #@param ["Qwen/Qwen2.5-3B", "Qwen/Qwen2.5-7B", "Qwen/Qwen3-8B-Base", "Qwen/Qwen3-4B-Base", "Qwen/Qwen2.5-14B", "microsoft/phi-2"]

!python scripts/run_rq1.py \
    --model {smoke_model} \
    --samples 2 \
    --methods occlusion,grad_x_input,ig,lime,shap \
    --max-evals 100 \
    --lime-samples 100 \
    --ig-steps 10 \
    --ig-baseline mean \
    --output-dir scratch/smoke_test_rq1 \
    --no-resume

### 4. Main RQ1 Stability Experiment (Milestone Auto-Resume: 310 -> 500 -> 872)
- Automatically saves checkpoints every 10 samples (`rq1_stability_raw.csv`, `rq1_perturbation_decisions.csv`, and `rq1_attributions.jsonl`).
- Prefix-preserving sampling (`seed=42`) ensures seamless continuation across milestones without recomputing completed samples.
- Default configuration runs `Qwen/Qwen2.5-3B` for `N = 310` with `ig_baseline = "mean"`.

In [ ]:
#@title Main RQ1 Stability Parameters
model = "Qwen/Qwen2.5-3B" #@param ["Qwen/Qwen2.5-3B", "Qwen/Qwen2.5-7B", "Qwen/Qwen3-8B-Base", "Qwen/Qwen3-4B-Base", "Qwen/Qwen2.5-14B", "microsoft/phi-2"]
milestones = "310" #@param ["310", "310,500", "300", "500", "872"]
seed = 42 #@param {type:"integer"}
methods = "occlusion,grad_x_input,ig,lime,shap" #@param {type:"string"}

shap_max_evals = 2500 #@param {type:"integer"}
lime_samples = 1000 #@param {type:"integer"}
ig_steps = 100 #@param {type:"integer"}
ig_baseline = "mean" #@param ["mean", "zero", "pad"]
sub_batch_size = 0 #@param {type:"integer"}

max_run_samples = 0 #@param {type:"integer"}
overwrite_methods = False #@param {type:"boolean"}
no_noise_floor = False #@param {type:"boolean"}
auto_shutdown = True #@param {type:"boolean"}

tag = model.split("/")[-1]
output_dir = os.path.join(DRIVE_RESULTS, f"rq1_stability_{tag}")
limit_flag = f"--max-run-samples {max_run_samples}" if max_run_samples > 0 else ""
overwrite_flag = "--overwrite-methods" if overwrite_methods else ""
noise_flag = "--no-noise-floor" if no_noise_floor else ""
sub_batch_flag = f"--sub-batch-size {sub_batch_size}" if sub_batch_size > 0 else ""

target_list = [int(x.strip()) for x in milestones.split(",") if x.strip()]

for target_n in target_list:
    print(f"\n{'='*75}")
    print(f"  RQ1 STABILITY MILESTONE: N = {target_n} | MODEL: {model} | IG: {ig_baseline}")
    print(f"  Output Directory: {output_dir}")
    print(f"{'='*75}\n")
    !python scripts/run_rq1.py \
        --model {model} \
        --samples {target_n} \
        --seed {seed} \
        --methods {methods} \
        --max-evals {shap_max_evals} \
        --lime-samples {lime_samples} \
        --ig-steps {ig_steps} \
        --ig-baseline {ig_baseline} \
        {sub_batch_flag} \
        {overwrite_flag} \
        {limit_flag} \
        {noise_flag} \
        --output-dir "{output_dir}"

if auto_shutdown and IN_COLAB_HOSTED:
    print("Releasing GPU and stopping Colab billing...")
    from google.colab import runtime
    runtime.unassign()

### 5. Results Summary Table (Stability, Preservation Rate & Perturbation Strength)

In [ ]:
import os
import pandas as pd
from IPython.display import display

raw_csv = os.path.join(output_dir, "rq1_stability_raw.csv")
if os.path.exists(raw_csv):
    df_raw = pd.read_csv(raw_csv)
    main_df = df_raw[(df_raw["perturbation_category"] != "control") & (df_raw["is_preserved"] == True)]
    ctrl_df = df_raw[df_raw["perturbation_category"] == "control"]

    overall = main_df.groupby("explainer_name").agg(
        n_pairs=("spearman", "size"),
        n_samples=("sample_id", "nunique"),
        spearman_mean=("spearman", "mean"),
        spearman_sem=("spearman", "sem"),
        kendall_mean=("kendall", "mean"),
        jaccard_k1_mean=("jaccard_k1", "mean"),
        jaccard_k3_mean=("jaccard_k3", "mean"),
        jaccard_k5_mean=("jaccard_k5", "mean"),
        time_s_mean=("orig_time_s", "mean"),
        fwd_passes_mean=("orig_forward_passes", "mean"),
        bwd_passes_mean=("orig_backward_passes", "mean"),
        vram_mb_max=("vram_mb", "max")
    ).reset_index()

    if len(ctrl_df) > 0:
        floor = ctrl_df.groupby("explainer_name").agg(
            noise_floor_spearman=("spearman", "mean"),
            noise_floor_jaccard_k3=("jaccard_k3", "mean")
        ).reset_index()
        overall = overall.merge(floor, on="explainer_name", how="left")

    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 1000)
    print("=== Overall Stability & Noise Floor ===")
    display(overall.round(3))

    pres_csv = os.path.join(output_dir, "rq1_preservation_summary.csv")
    if os.path.exists(pres_csv):
        print("\n=== Decision Preservation Rate per Perturbation ===")
        display(pd.read_csv(pres_csv).round(3))

    strength_csv = os.path.join(output_dir, "rq1_strength_analysis.csv")
    if os.path.exists(strength_csv):
        print("\n=== Stability vs. Perturbation Strength ===")
        display(pd.read_csv(strength_csv).round(3))
else:
    print(f"Raw file {raw_csv} does not exist yet.")

### 6. Stability Across Perturbation Categories & Boxplot Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if os.path.exists(raw_csv):
    df_raw = pd.read_csv(raw_csv)
    main_df = df_raw[(df_raw["perturbation_category"] != "control") & (df_raw["is_preserved"] == True)]

    cat_piv = main_df.pivot_table(
        index="explainer_name",
        columns="perturbation_category",
        values="spearman",
        aggfunc="mean"
    ).round(3)
    print("Mean Spearman Stability by Perturbation Category:")
    display(cat_piv)

    plt.figure(figsize=(10, 5.5), dpi=150)
    sns.boxplot(data=main_df, x="explainer_name", y="spearman", hue="perturbation_category")
    plt.title(f"RQ1: Attribution Ranking Stability by Perturbation Category – {tag}", fontweight="bold")
    plt.xlabel("XAI Method")
    plt.ylabel(r"Spearman Rank Correlation $\rho$")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "rq1_category_boxplot.png"))
    plt.show()

### 7. Manual Session Teardown (Stop Compute Units Billing)

In [ ]:
if IN_COLAB_HOSTED:
    from google.colab import runtime
    print("Disconnecting Colab runtime...")
    runtime.unassign()